### Init

In [1]:
import pandas as pd
from pathlib import Path
import pyarrow.feather as feather
import pyarrow.dataset as ds
import duckdb

root = Path.cwd().parent
# root = Path(__file__).resolve().parent.parent

neurons_file = root / 'data' / 'body-annotations-male-cns-v1.0-minconf-0.5.feather'
connections_file = root / 'data' / 'connectome-weights-male-cns-v1.0-minconf-0.5.feather'

#### Exploring Neurons

In [2]:
df = pd.read_feather(neurons_file)

print(df.head())    # a lot of data

print(df.columns.tolist()) # or just df.columns
# see if any columns are self explanatory or trash
# notes are written on the .md cheatsheet

print(df.shape)     # 211577 rows, 36 cols

   assignedOlHex1  assignedOlHex2  bodyId                      flywireType  \
0             NaN             NaN   10001                            DNp01   
1             NaN             NaN   10002                           OCG01d   
2             NaN             NaN   10003                              VCH   
3             NaN             NaN   10005                          AOTU019   
4             NaN             NaN   10006  VS1,VS2,VS3,VS4,VS5,VS6,VS7,VS8   

     group     instance somaSide            statusLabel          superclass  \
0  10001.0  DNp01(GF)_R        R         Roughly traced   descending_neuron   
1  10002.0     OCG01d_L        L         Roughly traced   visual_projection   
2  10003.0        VCH_R        R  Prelim Roughly traced  visual_centrifugal   
3  10005.0    AOTU019_R        R         Roughly traced        cb_intrinsic   
4  10006.0         VS_L        L  Prelim Roughly traced   visual_projection   

      type  ... entryNerve mancSerial mcnsSerial serialM

> the question is now, why is there so much rows (211k) compared to the claimed 166k number that online sources say?

In [3]:
# filtering neurons that have the traced status
traced = df[df['status'] == 'Traced']

# find what type of neurons are confirmed, and see how many there are
print(traced['type'].value_counts().head())

print(traced.shape)

type
Tm3    2054
T3     1940
T2a    1872
L5     1787
L2     1779
Name: count, dtype: int64
(165122, 36)


> Tm3, T3, T2a, L5, L2 are motion-detection neurons in the optic lobe.
> 
> can investigate more by changing the head value
> 
> the goal is to identify which neurons are for the specific use case
> 
> also, notice that there are 165k traced neurons, closer to that claimed 166k value online

In [4]:
one_neuron = traced.iloc[0]
print(one_neuron)

assignedOlHex1                             NaN
assignedOlHex2                             NaN
bodyId                                   10001
flywireType                              DNp01
group                                  10001.0
instance                           DNp01(GF)_R
somaSide                                     R
statusLabel                     Roughly traced
superclass                   descending_neuron
type                                     DNp01
vfbId                             VFB_jrmc30my
hemibrainType                      Giant Fiber
itoleeHl                      putative_primary
supertype                                  NaN
birthtime                                early
mancBodyid                             10000.0
mancGroup                              10000.0
mancType                                 DNp01
subclass                                    lt
synonyms          Kennedy and Broadie 2018: GF
class                                      NaN
rootSide     

> `rootSide` being NaN while `somaSide` is R is worth noting as a real data gap. a NaN here just means that particular technical anchor point wasn't populated for this neuron, doesn't say anything about the neuron itself, and instead about that column's completeness.
> 
> the one neuron's type is DNp01, it's part of a bigger circuit called the giant fiber (hence the GF in `instancetype`: `DNp01(GF)_R`) 
> 
> the giant fiber controls escape behaviours. it's particularly in a pair due to the _R postfix. to further confirm:

In [5]:
print(traced[traced['type'] == 'DNp01'])

   assignedOlHex1  assignedOlHex2  bodyId flywireType    group     instance  \
0             NaN             NaN   10001       DNp01  10001.0  DNp01(GF)_R   
6             NaN             NaN   10010       DNp01  10001.0  DNp01(GF)_L   

  somaSide     statusLabel         superclass   type  ... entryNerve  \
0        R  Roughly traced  descending_neuron  DNp01  ...        NaN   
6        L  Roughly traced  descending_neuron  DNp01  ...        NaN   

  mancSerial mcnsSerial serialMotif fruDsx  exitNerve  receptorType  \
0        NaN        NaN         NaN    NaN        NaN           NaN   
6        NaN        NaN         NaN    NaN        NaN           NaN   

            somaLocation tosomaLocation  status  
0  [37124, 22258, 36274]           None  Traced  
6  [58238, 22234, 36276]           None  Traced  

[2 rows x 36 columns]


> in the results above, DNp01 does in fact have a left and right side
>
> next step is to get the connectivity file, and filter to the interested neurons
> 
> that connectivity file is the "wiring" that connects the neurons

### Exploring Connections

> the goal is to filter the connections to something small to work with
> need to investigate the data via columns again

In [6]:
meta = feather.read_table(connections_file, memory_map=True)

> feather.read_table with memory map = True uses less memory for such a large file
>
> it reads the file's structure (column names, types, row count) off disk. it doesn't pull actual data into memory
>
> attempting to read such a file into pandas crashed the environment, so this lighter approach is used

In [7]:
print("schema:\n", meta.schema)
print("\nrows: ", meta.num_rows)

schema:
 body_pre: int64
body_post: int64
weight: int64
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 604

rows:  151856684


> looks to be 3 int columns but too big to load in full. body_pre, body_post, weight

In [8]:
# look at 5 rows
print(meta.slice(0, 5).to_pandas())

   body_pre  body_post  weight
0     10352      10351    2591
1     13612      16076    2443
2     10663      10051    2248
3     10289      10611    2230
4     10893      10066    1929


> 
> recognize that a neuron has a `bodyId`, so this table shows who talks to who
> 
> weight = synapse count. high numbers mean a large number of connections (information can travel efficiently between)
>
> next, need to tell what neuron is on either end of a connection via superclass, as superclass tells the general role

In [9]:
id_to_superclass = traced.set_index('bodyId')['superclass'].to_dict()
# this line makes bodyId the unique identifier for the df, and selects only the superclass column
# maps every single annotated neuron that is traced

# format is { id : superclass }
first_few = list(id_to_superclass.items())[:3]
print(first_few)

[(10001, 'descending_neuron'), (10002, 'visual_projection'), (10003, 'visual_centrifugal')]


> next is attaching meaning to these connections by mapping superclass -> superclass, rather than id -> id
> 
> but because the dataset is so big, a lazier loading method is used: .dataset -> it reads already-partitioned data (like a directory of files), allowing to stream or filter it without loading the whole thing.
>
> this works for the specific use case; the goal at that point was to peek at a sample, pull one chunk of rows just to see what the data looked like, confirm column names, eyeball a few real connections.
> 
> do not use .read_table because it will read a single file entirely into memory as an Arrow table. the whole table is not needed.
> 
> and read_feather materializes the data which is too expensive for a dataset this large.

In [ ]:
dataset = ds.dataset(connections_file, format="feather")

batches = dataset.to_batches(columns=["body_pre", "body_post", "weight"])
# this essentially will give the data in chunks, not all at once

first_batch = next(batches) # pull a batch

sample = first_batch.to_pandas()
print(sample.shape) # see how big that batch is

# add the translation columns
sample['pre_sc'] = sample['body_pre'].map(id_to_superclass)
sample['post_sc'] = sample['body_post'].map(id_to_superclass)

print(sample.head())

(65536, 3)
   body_pre  body_post  weight              pre_sc       post_sc
0     10352      10351    2591        ol_intrinsic  ol_intrinsic
1     13612      16076    2443        ol_intrinsic  ol_intrinsic
2     10663      10051    2248  visual_centrifugal  ol_intrinsic
3     10289      10611    2230  visual_centrifugal  ol_intrinsic
4     10893      10066    1929        ol_intrinsic  ol_intrinsic


> now readable connections are visible i.e., optic lobe neuron talking to another optic lobe neuron
>
> weights seem to be in descending order

In [11]:
# builds the small table DuckDB needs to translate bodyIds into superclasses
lookup = traced[['bodyId', 'superclass']]

# DuckDB does the join/aggregation in compiled C++ under the hood
# it never materializes 151M rows into Python objects at all
# it doesn't have to copy the data, so it seems it like works directly on the file
con = duckdb.connect()

# settings: caps how much memory DuckDB is allowed to use
con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA threads=2")

# key: it takes the dataframe as if it's a queryable table
con.register('lookup', lookup)
con.register('connections', meta)

# THIS WILL TAKE A BIT TO RUN
flow_df = con.execute("""
    SELECT pre.superclass AS pre_sc, post.superclass AS post_sc, SUM(c.weight) AS total_weight
    FROM connections AS c
    JOIN lookup AS pre ON c.body_pre = pre.bodyId
    JOIN lookup AS post ON c.body_post = post.bodyId
    GROUP BY pre_sc, post_sc
    ORDER BY total_weight DESC
""").df()

print(flow_df.head(30))

                pre_sc             post_sc  total_weight
0         ol_intrinsic        ol_intrinsic    35690242.0
1         cb_intrinsic        cb_intrinsic    34364111.0
2        vnc_intrinsic       vnc_intrinsic    11040961.0
3         ol_intrinsic   visual_projection     7714137.0
4    visual_projection        cb_intrinsic     3847546.0
5         cb_intrinsic   descending_neuron     2732690.0
6        vnc_intrinsic           vnc_motor     2214497.0
7          vnc_sensory       vnc_intrinsic     1967945.0
8     ascending_neuron        cb_intrinsic     1872657.0
9           cb_sensory        cb_intrinsic     1785077.0
10   descending_neuron       vnc_intrinsic     1718740.0
11       vnc_intrinsic    ascending_neuron     1636228.0
12   visual_projection   visual_projection     1613964.0
13    ascending_neuron       vnc_intrinsic     1594252.0
14  visual_centrifugal        ol_intrinsic     1469887.0
15   visual_projection        ol_intrinsic     1321728.0
16   descending_neuron        c

> a dominant pattern is that everything talks to itself:
>
> ol_intrinsic -> ol_intrinsic, cb_intrinsic -> cb_intrinsic, etc
>
> but other than that a pipeline is visible:
>
> ol_intrinsic > visual_projection > cb_instrinsic > descending_neuron > vnc_intrinsic
>
> translation: eyes > optic nerve > brain/decision > spinal cord > reflex/muscle

In [12]:
# stop and think
superclasses = traced['superclass'].unique()
print(superclasses)

<ArrowStringArray>
[    'descending_neuron',     'visual_projection',    'visual_centrifugal',
          'cb_intrinsic',          'ol_intrinsic',              'cb_motor',
      'ascending_neuron',          'cb_endocrine',            'cb_sensory',
         'vnc_intrinsic',    'sensory_descending',            'ol_sensory',
   'efferent_descending',     'sensory_ascending',          'vnc_efferent',
           'cb_efferent',                   'ENS',    'efferent_ascending',
                     nan,        'cb_sensory_tbc', 'visual_projection_tbc',
             'vnc_motor', 'sensory_ascending_tbc',           'vnc_sensory',
         'vnc_endocrine',       'vnc_sensory_tbc',               'vnc_tbc']
Length: 27, dtype: str


exploring_connections()

### Conclusion

new question: how would a fly hit an arrow on rhythm, what problems are there with that?
1. input representation - ol_sensory / ol_intrinsic
2. timing/beat - could the fly be a visual player or actually go with the beat? need to think about this
3. decisions - cb_intrinsic(thinking) / descending_neuron(tells what body should do)
4. motor output - vnc_motor to drive the legs, arms, wings
5. feedback/scoring - the fly should learn from misses

at the current time of writing this this is what should be understood:
- one file is about identity (what is this neuron)
- the other is about relationship (how does it talk to others)